# 05 蒙特卡洛 (Monte Carlo) 方法

**核心思想**：不需要环境模型，直接用 **完整回合的实际回报** 来估计 V/Q。

$$V^\pi(s) \approx \frac{1}{N(s)} \sum_{i=1}^{N(s)} G_t^{(i)}$$

**优点**：无偏估计、无需模型
**缺点**：必须等回合结束、方差大、只适用 episodic 任务

## 与定位的联系
在轨迹后处理（trajectory smoothing）任务中，我们能拿到完整轨迹和真值，类似于 MC 的设定。

In [ ]:
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt

np.random.seed(42)

GRID = 4
TERMINAL = [0, 15]
ACTIONS = ['U', 'D', 'L', 'R']
DELTAS = {'U': (-1, 0), 'D': (1, 0), 'L': (0, -1), 'R': (0, 1)}

def step(s, a):
    if s in TERMINAL:
        return s, 0.0, True
    r, c = s // GRID, s % GRID
    dr, dc = DELTAS[a]
    nr, nc = max(0, min(GRID-1, r+dr)), max(0, min(GRID-1, c+dc))
    s_next = nr * GRID + nc
    return s_next, -1.0, s_next in TERMINAL

## 1. 生成一个回合（episode）

In [ ]:
def generate_episode(policy, max_steps=100):
    s = np.random.choice([i for i in range(16) if i not in TERMINAL])
    episode = []
    for _ in range(max_steps):
        a = np.random.choice(ACTIONS, p=policy[s])
        s_next, r, done = step(s, a)
        episode.append((s, a, r))
        if done:
            break
        s = s_next
    return episode

uniform_policy = {s: np.ones(4) / 4 for s in range(16)}
ep = generate_episode(uniform_policy)
print('一个 episode (s, a, r):')
for st, ac, rw in ep[:10]:
    print(f'  s={st}, a={ac}, r={rw}')
print(f'... 共 {len(ep)} 步')

## 2. First-Visit MC 预测

**算法**：
1. 收集大量 episode
2. 对每个 episode：从后往前算 $G_t$
3. 对每个状态 s，**仅在 episode 中第一次访问 s 时**记录 $G_t$
4. $V(s) = \text{mean}(\text{所有记录})$

In [ ]:
def mc_prediction(policy, num_episodes=5000, gamma=1.0):
    returns = defaultdict(list)
    V = defaultdict(float)
    for ep_i in range(num_episodes):
        episode = generate_episode(policy)
        G = 0
        visited = set()
        for s, a, r in reversed(episode):
            G = gamma * G + r
            if s not in visited:
                returns[s].append(G)
                V[s] = np.mean(returns[s])
                visited.add(s)
    return V

V_mc = mc_prediction(uniform_policy, num_episodes=5000)
v_grid = np.zeros((GRID, GRID))
for s, v in V_mc.items():
    v_grid[s // GRID, s % GRID] = v
print('MC 估计的 V (随机策略):')
print(v_grid.round(1))

和上一节用 DP 求出的 V 对比，应该相近（误差来自采样）。

## 3. MC 控制：ε-greedy 改进策略

**ε-greedy**：以 1-ε 概率选当前最好的，ε 概率随机探索。

In [ ]:
def mc_control(num_episodes=20000, gamma=1.0, epsilon=0.1):
    Q = defaultdict(lambda: np.zeros(4))
    returns_count = defaultdict(int)
    
    def policy_action(s):
        if np.random.rand() < epsilon:
            return np.random.randint(4)
        return int(np.argmax(Q[s]))
    
    for ep_i in range(num_episodes):
        s = np.random.choice([i for i in range(16) if i not in TERMINAL])
        episode = []
        for _ in range(100):
            a_idx = policy_action(s)
            a = ACTIONS[a_idx]
            s_next, r, done = step(s, a)
            episode.append((s, a_idx, r))
            if done:
                break
            s = s_next
        
        G = 0
        visited = set()
        for s, a_idx, r in reversed(episode):
            G = gamma * G + r
            key = (s, a_idx)
            if key not in visited:
                returns_count[key] += 1
                Q[s][a_idx] += (G - Q[s][a_idx]) / returns_count[key]
                visited.add(key)
    return Q

Q_star = mc_control()
print('MC 控制学到的最优策略：')
for s in range(16):
    if s in TERMINAL:
        print('T', end=' ')
    else:
        print(ACTIONS[np.argmax(Q_star[s])], end=' ')
    if (s+1) % 4 == 0:
        print()

## 4. 思考

1. **为什么需要 ε-greedy？** —— 否则一旦初始 Q 偏向某个动作，就再也学不到其他动作的真实价值（**exploration 问题**）。
2. **MC 的局限**：必须等回合结束。如果是连续任务（无终止），MC 不适用。
3. **方差大**：单个 episode 的 G 可能差异很大，所以 MC 收敛慢。

👉 这些痛点正好引出了 **TD 方法**（下一节）。